## Imports and Environment

In [ ]:
# SimpleDirectoryReader is dynamic, detects file type and uses appropriate reader
from llama_index.core import VectorStoreIndex, Settings, PromptTemplate, StorageContext, SQLDatabase, SimpleDirectoryReader
from llama_index.core.utilities.sql_wrapper import SQLDatabase
from llama_index.core.query_engine import NLSQLTableQueryEngine, KnowledgeGraphQueryEngine
from llama_index.core.workflow import Workflow, StartEvent, StopEvent, step, Context, Event
from llama_index.core.retrievers import SQLRetriever
from llama_index.core.schema import TextNode
from llama_index.core.tools import QueryEngineTool
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.query_engine import NLSQLTableQueryEngine, RetrieverQueryEngine, SQLTableRetrieverQueryEngine
from llama_index.core.retrievers import SQLRetriever
from llama_index.core.objects import SQLTableNodeMapping, ObjectIndex, SQLTableSchema
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.openai import OpenAI
from llama_parse import LlamaParse

from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer, util

import pandas as pd, re, ast, textwrap
from sqlalchemy import create_engine, text, inspect

from datasets import Dataset

import ragas
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
    LLMSQLEquivalence
)
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas import evaluate, EvaluationDataset

from dotenv import load_dotenv, find_dotenv
from typing import Dict, Any, Tuple, Optional, List
import torch
import os
import re
import json
import csv
from tqdm import tqdm

# Project root path for Azure Sandpit environment
project_root_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone" 

# Change the current working directory to the project root
os.chdir(project_root_path)


# --- FIX 2: Bypass find_dotenv() and use a direct, verified path ---
dotenv_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env"

# Add a critical check to ensure the .env file exists at this path
if not os.path.exists(dotenv_path):
    raise FileNotFoundError(
        f"CRITICAL ERROR: .env file NOT FOUND at the expected path: {dotenv_path}\n"
        f"Please double-check the path you pasted into 'project_root_path'."
    )


# Load the .env file from the explicit, verified path
load_dotenv(dotenv_path=dotenv_path)

# The project root is now simply the current working directory
project_root = os.getcwd()

# --- Now, the rest of your variable loading will work correctly ---
relative_data_dir = os.getenv("SQL_DATASET_DIR")

# Add a check to make sure the variable was loaded successfully from the file
if not relative_data_dir:
    raise ValueError(
        "ERROR: 'SQL_DATASET_DIR' was not found in your .env file, or the file is empty."
    )

data_directory = os.path.join(project_root, relative_data_dir)

hf_token = os.getenv("HUGGINGFACE_TOKEN")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# --- Final Verification ---
print(f"✅ Project root successfully set to: {project_root}")
print(f"✅ .env file loaded from: {dotenv_path}")
print(f"📁 Data directory set to: {data_directory}")

## Helper Function to Sanitize File Names

In [ ]:
def sanitize_table_name(filename):
    """
    Cleans a filename to create a safe, SQL-compliant table name.
    - Converts to lowercase
    - Replaces spaces and hyphens with underscores
    - Removes all other non-alphanumeric characters (except underscores)
    """
    # Remove the .csv extension
    name = os.path.splitext(filename)[0]
    # Convert to lowercase and replace spaces/hyphens
    name = name.lower().replace(' ', '_').replace('-', '_')
    # Remove any remaining invalid characters
    name = re.sub(r'[^a-z0-9_]', '', name)
    return name

In [ ]:
# Create an in-memory SQLite database
# This database exists only as long as the script is running
engine = create_engine("sqlite:///:memory:")

# --- Dynamically load all CLEANED CSVs from the 'SQL_Dataset' directory ---
# This should point to the folder where your 'run_SQL_cleaning.py' script saved the files.
sql_data_directory = "SQL_Dataset" 
table_names = [] # To keep track of the tables we create

print(f"Searching for cleaned CSV files to ingest in '{sql_data_directory}'...")

# Check if the directory exists to avoid errors
if not os.path.isdir(sql_data_directory):
    print(f"Error: The directory '{sql_data_directory}' was not found. Please ensure the cleaning script ran successfully.")
else:
    for filename in os.listdir(sql_data_directory):
        if filename.endswith(".csv"):
            try:
                file_path = os.path.join(sql_data_directory, filename)
                
                # 1. Load the already-cleaned CSV into a DataFrame
                cleaned_df = pd.read_csv(file_path)
                
                # 2. Create a clean table name from the filename
                # Example: "cleaned_m891481.csv" -> "cleaned_m891481"
                table_name = sanitize_table_name(filename)
                table_names.append(table_name)
                
                # 3. Ingest the cleaned DataFrame into the SQL database
                cleaned_df.to_sql(table_name, engine, index=False, if_exists='replace')
                
                print(f" - Successfully ingested '{filename}' into SQL table '{table_name}'")
            except Exception as e:
                print(f" - FAILED to ingest {filename}. Error: {e}")

print(f"\nIn-memory SQL database created and populated with {len(table_names)} table(s).")
sql_database = SQLDatabase(engine)


## Llama 3.1 8B Instruct

In [ ]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Initialize the tokenizer to get the token ID for stop sequence
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
# The semicolon is the stop character, get its token ID
semicolon_token_id = tokenizer.convert_tokens_to_ids(";")

# Initialize the LLM with the correct stop condition
llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    device_map="auto",
    model_kwargs={"token": hf_token, "dtype": torch.bfloat16},
    # Use 'eos_token_id' which is the correct parameter for this purpose
    generate_kwargs={
        "temperature": 0.1,
        "do_sample": True,
        # Stop generating as soon as it outputs a semicolon
        "eos_token_id": semicolon_token_id,
    }
)

print("HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.")

## Query Time Table Retrieval

In [ ]:
inspector = inspect(engine)

# Create SQLTableNodeMapping and ObjectIndex for table retrieval
table_node_mapping = SQLTableNodeMapping(sql_database)
all_table_schema_objs = [
    SQLTableSchema(table_name=name) for name in inspector.get_table_names()
]

# Create a vector index over the table schemas for retrieval
obj_index = ObjectIndex.from_objects(
    all_table_schema_objs,
    table_node_mapping,
    index_cls=VectorStoreIndex,
)
obj_retriever = obj_index.as_retriever(similarity_top_k=3)

# Use the object retriever to dynamically find the
# right tables to use based on the user's query
query_engine_table_retrieval = SQLTableRetrieverQueryEngine(
    sql_database, obj_retriever
)

# --- Example Usage for Table Retrieval ---
print("\n--- Testing Text-to-SQL with Query-Time Table Retrieval ---")
query_1 = "In 2010, what was the number of male inmates?"
response_1 = query_engine_table_retrieval.query(query_1)
print(f"Query: {query_1}")
print(f"Response: {response_1}\n")
print(f"Generated SQL: {response_1.metadata['sql_query']}\n")

## Query Time Row Retrieval

In [ ]:
# -Create Vector Indices for Rows in Each Table
table_names = inspector.get_table_names()
table_row_indices: Dict[str, VectorStoreIndex] = {}
table_row_query_engines: Dict[str, RetrieverQueryEngine] = {}

print("\n--- Starting Row-Level Indexing for Each Table ---")
for table_name in table_names:
    print(f"Processing table: {table_name}")
    # Read table into pandas DataFrame
    df = pd.read_sql_table(table_name, engine)

    # Create TextNode objects for each row
    row_nodes: List[TextNode] = []
    for i, row in df.iterrows():
        # Combine all columns of the row into a single text string
        row_text = " | ".join(map(str, row.values))
        node = TextNode(
            text=f"Row for table '{table_name}': {row_text}",
            metadata={"table_name": table_name, "row_index": i},
        )
        row_nodes.append(node)

    # Create a VectorStoreIndex from the row nodes
    row_index = VectorStoreIndex(row_nodes)
    table_row_indices[table_name] = row_index

    # Create a query engine for this table's rows
    table_row_query_engines[table_name] = row_index.as_query_engine(
        similarity_top_k=5
    )
print("--- Row-Level Indexing Complete ---\n")

# Create QueryEngineTools for Each Table's Row Data
query_engine_tools: List[QueryEngineTool] = []
for table_name, query_engine in table_row_query_engines.items():
    tool_metadata = f"This tool provides access to the rows of the '{table_name}' table. Use it to find specific values or examples within the table."
    tool = QueryEngineTool.from_defaults(
        query_engine=query_engine,
        name=f"row_retriever_{table_name}",
        description=tool_metadata,
    )
    query_engine_tools.append(tool)

# This engine combines both table retrieval and row retrieval.
# obj_retriever` finds the right tables
# tools (row retrievers) help find the right values within those tables

final_query_engine = SQLTableRetrieverQueryEngine(
    sql_database,
    obj_retriever,
    tools=query_engine_tools, # Tools for row-level retrieval
)

# --- Example Usage for Row Retrieval ---
print("--- Testing Text-to-SQL with Query-Time Row-Level Retrieval ---")
query_2 = "In the table for education level, what was the population for those with 'No Formal Education / Lower Primary' in 2021?"
response_2 = final_query_engine.query(query_2)
print(f"Query: {query_2}")
print(f"Response: {response_2}\n")
print(f"Generated SQL: {response_2.metadata['sql_query']}")

## Benchmarking

In [ ]:
# --- Configuration ---
relative_benchmark_path = os.getenv("SQL_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("SQL_BENCHMARK_DATASET_DIR not set in .env")

BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "adv_sql_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load Data ---
print(f"Loading benchmark data from {BENCHMARK_FILE_PATH}...")
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding="latin1")

# Slice the DataFrame to get only the first 3 rows for a quick test
# benchmark_df = benchmark_df.head(3)

# --- Prepare DataFrame for Ragas ---
if 'gt_answer' in benchmark_df.columns:
    benchmark_df['gt_answer'] = benchmark_df['gt_answer'].fillna('')
    benchmark_df = benchmark_df.rename(columns={"gt_answer": "ground_truth"})

if 'gt_query' in benchmark_df.columns:
    benchmark_df['gt_query'] = benchmark_df['gt_query'].fillna('')
    benchmark_df = benchmark_df.rename(columns={"gt_query": "ground_truths"})
    benchmark_df['ground_truths'] = benchmark_df['ground_truths'].apply(lambda x: [x] if isinstance(x, str) else [])
else:
    raise ValueError("'gt_query' column not found in the benchmark file.")

print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")


# --- Generate Predictions and Collect Data ---
print("--- Running pipeline and collecting data for evaluation ---")
results_data = []
for index, row in tqdm(benchmark_df.iterrows(), total=benchmark_df.shape[0]):
    question = row['question']

    # Get the SQL query using the engine that reliably produces it in metadata
    sql_response = query_engine_table_retrieval.query(question)
    generated_sql = sql_response.metadata.get('sql_query', 'No SQL Query Generated')

    # Get the final more accurate answer
    final_response = final_query_engine.query(question)
    generated_answer = str(final_response)
    
    contexts = [node.get_content() for node in final_response.source_nodes]
    
    results_data.append({
        "question": question,
        "answer": generated_answer,
        "generated_sql": generated_sql,
        "contexts": contexts,
        "ground_truth": row.get('ground_truth'),
        "ground_truths": row.get('ground_truths')
    })

results_df = pd.DataFrame(results_data)
ragas_dataset = Dataset.from_pandas(results_df)

# --- Configure Ragas Metrics ---
rag_metrics = [
    answer_relevancy,
    faithfulness,
    context_precision,
    context_recall,
]
print("Initializing the judge LLM for SQL evaluation...")
judge_llm = llm_factory(model="gpt-4o")
sql_equivalence_metric = LLMSQLEquivalence(llm=judge_llm)

# --- Run Evaluations ---
print("Evaluating RAG metrics (answer_relevancy, faithfulness, etc.)...")
rag_result = evaluate(
    dataset=ragas_dataset,
    metrics=rag_metrics,
)
rag_scores_df = rag_result.to_pandas()
print("RAG evaluation complete.")

# --- Evaluate SQL Correctness ---
print("Evaluating SQL equivalence...")

sql_result = evaluate(
    dataset=ragas_dataset, 
    metrics=[sql_equivalence_metric],
    column_map={
        "answer": "generated_sql",      
        "ground_truths": "ground_truths", 
        "reference": "ground_truth",    
        "reference_contexts": "contexts"
    }
)
sql_scores_df = sql_result.to_pandas()
print("SQL evaluation complete.")


# --- Format and Save Final Results ---
# Merge scores from both evaluations into the results dataframe
final_df = results_df.copy()
rag_metric_names = [m.name for m in rag_metrics]
final_df = final_df.join(rag_scores_df[rag_metric_names])

sql_metric_name = sql_equivalence_metric.name
final_df = final_df.join(sql_scores_df[[sql_metric_name]])


# --- Print Overall Performance Metrics First ---
print("\n--- Overall Performance Metrics ---")
all_metric_names = rag_metric_names + [sql_metric_name]
print(final_df[all_metric_names].mean(numeric_only=True))
print("----------------------------------")


# --- Prepare the DataFrame for Final CSV Output ---
# Define the exact columns in final csv file
columns_to_keep = [
    "question",
    "ground_truth",         # Will be renamed to gt_answer
    "ground_truths",        # Will be renamed to gt_query
    "answer",               # Will be renamed to generated_answer
    "generated_sql",        
    "answer_relevancy",
    "faithfulness",
    "context_recall",
    "context_precision",
    sql_metric_name
]

# Overwrite the final_df with a new dataframe containing only the desired columns
final_df = final_df[columns_to_keep].copy()

final_df.rename(columns={
    "ground_truth": "gt_answer",
    "ground_truths": "gt_query",
    "answer": "generated_answer"
}, inplace=True)

# Save the final, clean dataframe to the single CSV file
final_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nClean benchmark results saved to {OUTPUT_FILE_PATH}")

Loading benchmark data from /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/Benchmark Dataset/sql_benchmark.csv...
Loaded 100 question-answer pairs for evaluation.
--- Running pipeline and collecting data for evaluation ---


  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [08:12<00:00,  4.92s/it]


Initializing the judge LLM for SQL evaluation...
Evaluating RAG metrics (answer_relevancy, faithfulness, etc.)...


Evaluating:   0%|          | 0/400 [00:00<?, ?it/s]

Exception raised in Job[12]: IndexError(list index out of range)
Exception raised in Job[4]: IndexError(list index out of range)
Exception raised in Job[8]: IndexError(list index out of range)
Exception raised in Job[0]: IndexError(list index out of range)
Exception raised in Job[20]: IndexError(list index out of range)
Exception raised in Job[24]: IndexError(list index out of range)
Exception raised in Job[16]: IndexError(list index out of range)
Exception raised in Job[28]: IndexError(list index out of range)
Exception raised in Job[40]: IndexError(list index out of range)
Exception raised in Job[32]: IndexError(list index out of range)
Exception raised in Job[44]: IndexError(list index out of range)
Exception raised in Job[48]: IndexError(list index out of range)
Exception raised in Job[52]: IndexError(list index out of range)
Exception raised in Job[56]: IndexError(list index out of range)
Exception raised in Job[60]: IndexError(list index out of range)
Exception raised in Job[84]:

RAG evaluation complete.
Evaluating SQL equivalence...


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

SQL evaluation complete.

--- Overall Performance Metrics ---
answer_relevancy                      0.921718
faithfulness                          0.075000
context_precision                     0.760000
context_recall                        0.720000
llm_sql_equivalence_with_reference    0.740000
dtype: float64
----------------------------------

Clean benchmark results saved to /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/adv_sql_benchmark_results.csv
